# S01 — Data Exploration & QA

Visual quality assurance on the extracted Spain grid hourly dataset.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_parquet("../data/spain_grid_hourly.parquet")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} → {df.index.max()}")
print(f"\nMissing values:")
print(df.isnull().sum())
df.describe().round(0)

Shape: (45480, 11)
Date range: 2020-12-31 23:00:00+00:00 → 2026-03-10 22:00:00+00:00

Missing values:
actual_demand_mw         136
forecast_demand_mw       108
gen_wind_mw                0
gen_solar_pv_mw            0
gen_hydro_mw               0
gen_combined_cycle_mw      0
gen_nuclear_mw             0
gen_total_mw               0
forecast_wind_mw          24
forecast_solar_mw         24
local_hour                 0
dtype: int64


,actual_demand_mw,forecast_demand_mw,gen_wind_mw,gen_solar_pv_mw,gen_hydro_mw,gen_combined_cycle_mw,gen_nuclear_mw,gen_total_mw,forecast_wind_mw,forecast_solar_mw,local_hour
count,45344.0,45372.0,45480.0,45480.0,45480.0,45480.0,45480.0,45480.0,45456.0,45456.0,45480.0
mean,28766.0,28712.0,168918.0,98704.0,79927.0,131362.0,147359.0,626270.0,168969.0,98688.0,12.0
std,4636.0,4677.0,87916.0,48542.0,41259.0,68487.0,24284.0,75681.0,87912.0,48550.0,7.0
min,2101.0,1167.0,17067.0,6752.0,12758.0,33658.0,152.0,341214.0,17067.0,6752.0,0.0
25%,24958.0,24918.0,101421.0,60812.0,46889.0,81290.0,128780.0,571695.0,101590.0,60812.0,6.0
50%,28864.0,28805.0,152513.0,89583.0,70782.0,112825.0,151967.0,625593.0,152566.0,89529.0,12.0
75%,32215.0,32163.0,221302.0,130478.0,106242.0,166883.0,167038.0,679490.0,221302.0,130478.0,17.0
max,43894.0,43863.0,456780.0,239688.0,212590.0,396454.0,171349.0,880899.0,456780.0,239688.0,23.0


## 1. Demand Curves

In [2]:
# Sample 1 month for detailed view
sample = df.loc["2024-01"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=sample.index, y=sample["actual_demand_mw"],
                         name="Actual Demand", line=dict(width=1)))
fig.add_trace(go.Scatter(x=sample.index, y=sample["forecast_demand_mw"],
                         name="Forecast Demand", line=dict(width=1, dash="dash")))
fig.update_layout(title="Demand: Actual vs Forecast (Jan 2024)",
                  yaxis_title="MW", height=400)
fig.show()

## 2. Generation Mix

In [3]:
gen_cols = [c for c in df.columns if c.startswith("gen_") and c != "gen_total_mw"]

# Monthly average generation by technology
monthly = df[gen_cols].resample("ME").mean()

fig = go.Figure()
for col in gen_cols:
    label = col.replace("gen_", "").replace("_mw", "").replace("_", " ").title()
    fig.add_trace(go.Scatter(x=monthly.index, y=monthly[col],
                             name=label, stackgroup="one"))
fig.update_layout(title="Monthly Average Generation Mix",
                  yaxis_title="MW", height=500)
fig.show()

## 3. Yearly Demand Distribution

In [4]:
df["year"] = df.index.year
fig = px.box(df, x="year", y="actual_demand_mw",
             title="Hourly Demand Distribution by Year",
             labels={"actual_demand_mw": "MW", "year": "Year"})
fig.update_layout(height=400)
fig.show()

## 4. Daily Profile by Hour

In [5]:
hourly_avg = df.groupby("local_hour")[["actual_demand_mw"]].mean()

fig = px.bar(hourly_avg, y="actual_demand_mw",
             title="Average Demand by Local Hour (CET/CEST)",
             labels={"actual_demand_mw": "MW", "local_hour": "Hour"})
fig.update_layout(height=400)
fig.show()

## 5. Missing Data Heatmap

In [6]:
# Monthly missing % per column
missing_monthly = df.resample("ME").apply(lambda x: x.isnull().mean() * 100)
cols_to_check = [c for c in df.columns if c not in ["local_hour", "year"]]

fig = px.imshow(missing_monthly[cols_to_check].T,
                title="Monthly Missing Data (%)",
                labels=dict(x="Month", y="Column", color="% Missing"),
                color_continuous_scale="Reds",
                aspect="auto")
fig.update_layout(height=400)
fig.show()

## 6. Forecast vs Actual Wind/Solar

In [7]:
sample = df.loc["2024-06"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Wind", "Solar PV"])

fig.add_trace(go.Scatter(x=sample.index, y=sample["gen_wind_mw"],
                         name="Actual Wind", line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=sample.index, y=sample["forecast_wind_mw"],
                         name="Forecast Wind", line=dict(width=1, dash="dash")), row=1, col=1)

fig.add_trace(go.Scatter(x=sample.index, y=sample["gen_solar_pv_mw"],
                         name="Actual Solar", line=dict(width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=sample.index, y=sample["forecast_solar_mw"],
                         name="Forecast Solar", line=dict(width=1, dash="dash")), row=2, col=1)

fig.update_layout(title="Actual vs Forecast (24h-lagged) — Jun 2024",
                  height=600, yaxis_title="MW", yaxis2_title="MW")
fig.show()

## Summary

Key observations from the QA:
- Demand data has expected daily/weekly seasonality
- Generation mix shows renewable growth over time
- Missing data is minimal and concentrated at API gap boundaries
- Wind/solar forecasts (24h-lagged) track actuals with expected delay